In [53]:
from langgraph.graph import StateGraph, START, END
from langchain_openrouter import ChatOpenRouter
from langchain_core.messages import SystemMessage, HumanMessage
from typing import TypedDict
from dotenv import load_dotenv
import os
import requests


In [54]:
load_dotenv()  # Load environment variables from .env file
api_key = os.getenv("open_router_key")
open_router_model = os.getenv("open_router_model")

In [55]:
from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(
    model="openrouter/free",
    temperature=0,
    max_tokens=1024,
    max_retries=2,
    # other params...
)

In [56]:
# create a state
class LLMState(TypedDict):
    question: str
    answer: str

In [57]:
def llm_qa(state: LLMState) -> LLMState:
    # extrate the question from state
    question = state['question']
    # form a prompt
    prompt = f'Answer the Following Question {question}'
    # ask that question to the llm
    answer = model.invoke(prompt).content
    # update teh answer in state
    state['answer'] = answer

    return state

In [58]:
# create a graph
qa_graph = StateGraph(LLMState)

# add nodes
qa_graph.add_node('llm_qa', llm_qa)

# connect edges
qa_graph.add_edge(START, 'llm_qa')
qa_graph.add_edge('llm_qa', END)

# compile graph
qa_workflow = qa_graph.compile()
# execute graph

In [61]:
## execute 

inital_state = {'question': 'How to be a millinaire in india'}

final_state = qa_workflow.invoke(inital_state)

print(final_state['answer'])

Becoming a millionaire in India (defined as having a net worth of ₹1 crore or more, depending on context) requires a combination of strategic planning, disciplined financial habits, and leveraging opportunities in India's growing economy. Here's a structured approach:

---

### **1. Define Your Goal Clearly**
- **Understand the Target**: A "millionaire" in India typically means a net worth of ₹1 crore (≈ $125,000 USD) or more. Clarify if you aim for liquid assets, real estate, or business equity.
- **Set a Timeline**: Wealth accumulation takes time. Set realistic milestones (e.g., 10–20 years) based on your income and savings rate.

---

### **2. Build High-Income Skills**
- **Focus on In-Demand Fields**:
  - **Technology**: Coding (Python, AI/ML), cybersecurity, cloud computing.
  - **Digital Marketing**: SEO, social media strategy, data analytics.
  - **Finance**: Investment analysis, fintech, stock trading.
  - **Healthcare**: Medical specializations, telemedicine, wellness startups

In [60]:
messages = [
    (
        "system",
        "You are a helpful assistant that translates English to Hindi. Translate the user sentence.",
    ),
    ("human", "I love programming."),
]
ai_msg = model.invoke(messages)
ai_msg.content

'मैं प्रोग्रामिंग को प्यार करता हूँ।\n'